# Demand Forecasting Experiments

This notebook experiments with different forecasting models and evaluates their performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path().absolute().parent / 'src'))

from pipeline import DataPipeline
from forecast import DemandForecaster, forecast_next_week
from risk import RiskPredictor

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Load Data

In [ ]:
# Load processed data
pipeline = DataPipeline()
df = pipeline.run_pipeline()

print(f"Dataset shape: {df.shape}")
print(f"Number of SKUs: {df['SKU'].nunique()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

## Model Comparison

In [ ]:
# Define models to test
models = ['random_forest', 'lightgbm']
results = {}

for model_type in models:
    print(f"\n{'='*60}")
    print(f"Testing {model_type} model")
    print(f"{'='*60}")
    
    try:
        forecaster = DemandForecaster(model_type=model_type)
        
        # Perform backtesting
        backtest_results = forecaster.rolling_origin_backtest(df, n_splits=5)
        
        results[model_type] = backtest_results
        
        print(f"Model WAPE: {backtest_results['avg_model_wape']:.2f}%")
        print(f"Baseline WAPE: {backtest_results['avg_baseline_wape']:.2f}%")
        print(f"Improvement: {backtest_results['avg_improvement']:.2f}%")
        
    except Exception as e:
        print(f"Error with {model_type}: {e}")

## Model Performance Comparison

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(2)
comparison_df

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Model WAPE
comparison_df['avg_model_wape'].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Model WAPE')
axes[0].set_ylabel('WAPE (%)')
axes[0].tick_params(axis='x', rotation=45)

# Baseline WAPE
comparison_df['avg_baseline_wape'].plot(kind='bar', ax=axes[1], color='lightcoral')
axes[1].set_title('Baseline WAPE')
axes[1].set_ylabel('WAPE (%)')
axes[1].tick_params(axis='x', rotation=45)

# Improvement
comparison_df['avg_improvement'].plot(kind='bar', ax=axes[2], color='lightgreen')
axes[2].set_title('Improvement over Baseline')
axes[2].set_ylabel('Improvement (%)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Generate Forecasts with Best Model

In [ ]:
# Select best model (lowest WAPE)
best_model = comparison_df['avg_model_wape'].idxmin()
print(f"Best model: {best_model}")

# Generate forecasts
forecast_df = forecast_next_week(df, model_type=best_model)

print(f"\nForecasts generated for {len(forecast_df)} SKUs")
forecast_df.head()

## Forecast Analysis

In [ ]:
# Forecast distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

forecast_df['Forecast'].hist(bins=50, ax=axes[0], color='skyblue')
axes[0].set_title('Forecast Distribution')
axes[0].set_xlabel('Forecasted Units')
axes[0].set_ylabel('Frequency')

forecast_df['Current_Stock'].hist(bins=50, ax=axes[1], color='lightcoral')
axes[1].set_title('Current Stock Distribution')
axes[1].set_xlabel('Current Stock')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 forecasts
top_forecasts = forecast_df.nlargest(10, 'Forecast')
print("Top 10 Forecasts:")
print(top_forecasts[['SKU', 'Category', 'Current_Stock', 'Forecast']])

## Risk Prediction

In [ ]:
# Predict risks
risk_predictor = RiskPredictor()
risk_df = risk_predictor.predict_risks(forecast_df)

print(f"Risk predictions completed for {len(risk_df)} SKUs")
risk_df.head()

In [ ]:
# Risk summary
risk_summary = risk_predictor.get_risk_summary(risk_df)
print("Risk Summary:")
for key, value in risk_summary.items():
    print(f"  {key}: {value}")

## Risk Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stockout risk
stockout_counts = risk_df['Stockout_Risk'].value_counts()
stockout_counts.plot(kind='bar', ax=axes[0], color=['red', 'orange', 'yellow', 'green'])
axes[0].set_title('Stockout Risk Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Overstock risk
overstock_counts = risk_df['Overstock_Risk'].value_counts()
overstock_counts.plot(kind='bar', ax=axes[1], color=['red', 'orange', 'yellow', 'green'])
axes[1].set_title('Overstock Risk Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Recommendation Analysis

In [ ]:
# Recommendation distribution
rec_counts = risk_df['Recommendation'].value_counts()
rec_counts.plot(kind='pie', autopct='%1.1f%%', figsize=(8, 8),
                 colors=['red', 'orange', 'blue', 'green'])
plt.title('Recommendation Distribution')
plt.ylabel('')
plt.show()

In [ ]:
# High priority actions (Reorder Now)
reorder_skus = risk_df[risk_df['Recommendation'] == 'Reorder Now']
print(f"SKUs requiring immediate reorder: {len(reorder_skus)}")
print(f"\nTop 10 SKUs to reorder:")
print(reorder_skus.nlargest(10, 'Risk_Score')[['SKU', 'Category', 'Current_Stock', 'Forecast', 'Risk_Score']])

## Category-wise Analysis

In [ ]:
from risk import analyze_inventory_health

category_health = analyze_inventory_health(risk_df)
category_health

## Save Results

In [ ]:
# Save forecasts and risks
forecast_df.to_csv('../data/processed/forecasts.csv', index=False)
risk_df.to_csv('../data/processed/risks.csv', index=False)

print("Results saved to data/processed/")

## Key Findings

### Model Performance:
- Best performing model: [To be filled after analysis]
- WAPE achieved: [To be filled after analysis]
- Improvement over baseline: [To be filled after analysis]

### Risk Insights:
- High stockout risk SKUs: [To be filled after analysis]
- High overstock risk SKUs: [To be filled after analysis]
- Categories requiring attention: [To be filled after analysis]

### Recommendations:
- Immediate actions needed: [To be filled after analysis]
- Long-term strategy: [To be filled after analysis]